# 10 Stage 2 Random Forest — Health Outcome Prediction

Targets: `target_unmet_fp` (and `target_anc_gap` when m14 is available)  
Depends on: `07_data_integration.ipynb`, `08_clustering.ipynb`

In [ ]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models.stage2_random_forest import (
    configure_logging,
    train_all_stage2_random_forest,
)
from src.models.stage2_xgboost import STAGE2_TARGETS, TARGET_DISPLAY, load_stage2_data

configure_logging()

In [ ]:
# Verify required Stage 2 input files exist before training
stage2_files = [
    PROJECT_ROOT / 'data/processed/stage2/X_stage2_preclustering.csv',
    PROJECT_ROOT / 'data/processed/stage2/y_stage2_targets.csv',
    PROJECT_ROOT / 'outputs/stage2_results/cluster_assignments.csv',
]
if not all(p.exists() for p in stage2_files):
    raise FileNotFoundError(
        'Run scripts/run_stage2_data_prep.py or 08_clustering.ipynb first.'
    )
print('Stage 2 inputs found.')

In [ ]:
# Inspect feature matrix and target availability
X_full, y = load_stage2_data()
print(f'Feature matrix (with cluster dummies): {X_full.shape}')
for col in y.columns:
    nn  = y[col].notna().sum()
    pos = int(y.loc[y[col].notna(), col].sum()) if nn else 0
    print(f'{col}: non-null N = {nn:,} | positive = {pos:,}')

In [ ]:
# Train separate Random Forest models per target (Stage 2 guide §3.2)
results = train_all_stage2_random_forest()
metrics_df = results.get('metrics_df')
if metrics_df is not None:
    display_cols = [
        'Target', 'TrainSize', 'TestSize',
        'ROC-AUC', 'F1-Score', 'CV_ROC-AUC', 'Barrier_Uplift',
    ]
    display(metrics_df[[c for c in display_cols if c in metrics_df.columns]])

In [ ]:
# Top feature importances per target
for target_col in STAGE2_TARGETS:
    if target_col not in results.get('targets', {}):
        print(
            f'Skipped {TARGET_DISPLAY.get(target_col, target_col)} '
            f'(no analytic sample)'
        )
        continue
    feat_df = results['targets'][target_col]['feature_importance']
    print(f'\n=== {TARGET_DISPLAY.get(target_col, target_col)} — top feature importances ===')
    print(feat_df[['feature_display', 'importance']].head(10).to_string(index=False))

In [ ]:
# Classification report, barrier uplift, and saved output summary
from pathlib import Path
import json

results_dir = PROJECT_ROOT / 'outputs' / 'stage2_results'

for target_col in STAGE2_TARGETS:
    if target_col not in results.get('targets', {}):
        continue

    display_name = TARGET_DISPLAY.get(target_col, target_col)
    print(f'\n{'='*60}')
    print(f'  {display_name}')
    print(f'{'='*60}')

    # Classification report
    cr_path = results_dir / f'classification_report_{target_col}.csv'
    if cr_path.exists():
        print('\nClassification Report:')
        display(pd.read_csv(cr_path).set_index('class'))

    # Barrier uplift
    bu_path = results_dir / f'rf_barrier_uplift_{target_col}.csv'
    if bu_path.exists():
        print('\nBarrier Uplift:')
        display(pd.read_csv(bu_path))

    # Hyperparameters
    hp_path = results_dir / f'best_hyperparameters_{target_col}.json'
    if hp_path.exists():
        print('\nBest Hyperparameters:')
        print(json.dumps(json.loads(hp_path.read_text()), indent=2))

print('\nAll non-graph outputs saved to:', results_dir)
non_graph = [p.name for p in sorted(results_dir.iterdir())
             if p.is_file() and p.suffix not in ('.png', '.svg', '.pdf', '.pkl')]
for name in non_graph:
    print(' ', name)